# Mixture of Markov Chains with EM

This notebook presents a cleaned version of the **Three Weather Stations** project from a graphical models coursework portfolio.

## Goal
Model symbolic weather sequences using a **mixture of first-order Markov chains** with latent station assignments, and estimate parameters via the **Expectation-Maximization (EM)** algorithm.

## What this notebook demonstrates
- latent-variable sequence modeling
- EM updates for mixture models
- numerical stability using log-domain computations
- sensitivity to initialization and the need for multi-start optimization

## Data
Place `meteo1.csv` in the same working directory as this notebook before running the final cell.

## Implementation

The code below loads the sequence data, defines the EM model, fits the model with multiple random restarts, and prints:
- the best log-likelihood
- mixture weights
- initial distributions
- transition matrices
- posterior responsibilities for the first few sequences

In [ ]:
from scipy.special import logsumexp

# Module 1: Robust Data Loading

def load_weather_data(filename):
    """
    Robustly loads the meteo1.csv file.
    Fixes the SyntaxWarning by using raw string r'\s+'.
    """
    try:
        df = pd.read_csv(filename, sep=r'\s+', header=None)
        data = df.values.astype(int)
        print(f"[Data Loaded] Source: {filename}")
        print(f"[Stats] Sequences: {data.shape[0]}, TimeSteps: {data.shape[1]}")
        return data
    except Exception as e:
        print(f"[Error] Failed to load data: {e}")
        return None

# Module 2: EM Algorithm Core with Smoothing
class MixtureMarkovEM:
    def __init__(self, n_components=3, n_states=3, max_iter=100, tol=1e-6,
                 random_state=None, smoothing=1e-6):
        self.K = n_components
        self.S = n_states
        self.max_iter = max_iter
        self.tol = tol
        self.rng = np.random.RandomState(random_state)
        self.smoothing = smoothing # Pseudocount
        self.final_ll_ = -np.inf

    def _init_params(self):
        # 1. Omega (Mixture Weights): Uniform + slight noise
        omega = np.ones(self.K) / self.K
        # Add tiny noise to break exact symmetry if needed, but Dirichlet is better
        self.log_omega = np.log(omega)

        # 2. Pi (Initial Probs): Random Dirichlet
        pi = self.rng.dirichlet(alpha=np.ones(self.S)*5, size=self.K)
        self.log_pi = np.log(pi + 1e-12)

        # 3. A (Transitions): Random Dirichlet
        A = self.rng.dirichlet(alpha=np.ones(self.S)*5, size=(self.K, self.S))
        self.log_A = np.log(A + 1e-12)

    def _e_step(self, X):
        N, T = X.shape
        log_L_cond = np.zeros((N, self.K))

        for k in range(self.K):
            # Init state contribution
            log_L_cond[:, k] = self.log_pi[k, X[:, 0]]
            # Transition contribution
            for t in range(T - 1):
                from_s = X[:, t]
                to_s = X[:, t+1]
                log_L_cond[:, k] += self.log_A[k, from_s, to_s]

        # Log Joint & Log Marginal
        log_joint = log_L_cond + self.log_omega
        log_marginal = logsumexp(log_joint, axis=1)

        # Log Gamma (Posterior)
        log_gamma = log_joint - log_marginal[:, np.newaxis]
        gamma = np.exp(log_gamma)

        return gamma, np.sum(log_marginal)

    def fit(self, X):
        self._init_params()
        log_lik_history = []

        for it in range(self.max_iter):
            # E-Step
            gamma, current_log_lik = self._e_step(X)
            log_lik_history.append(current_log_lik)

            # Convergence Check (Relative Error)
            if it > 0:
                prev_ll = log_lik_history[-2]
                rel_change = abs(current_log_lik - prev_ll) / (abs(prev_ll) + 1e-10)
                if rel_change < self.tol:
                    break

            # M-Step (with Smoothing)
            # 1. Update Omega
            Nk = gamma.sum(axis=0)
            # Add smoothing to weights to prevent 0
            self.log_omega = np.log((Nk + self.smoothing) / (len(X) + self.K * self.smoothing))

            # 2. Update Pi
            new_pi = np.zeros((self.K, self.S))
            for s in range(self.S):
                mask = (X[:, 0] == s)
                new_pi[:, s] = gamma[mask].sum(axis=0)

            # Normalize with smoothing
            new_pi += self.smoothing
            new_pi /= new_pi.sum(axis=1, keepdims=True)
            self.log_pi = np.log(new_pi)

            # 3. Update A
            new_A = np.zeros((self.K, self.S, self.S))
            for a in range(self.S):
                for b in range(self.S):
                    # Vectorized count per sequence
                    count_per_seq = np.zeros(len(X))
                    for t in range(X.shape[1] - 1):
                         count_per_seq += ((X[:, t] == a) & (X[:, t+1] == b))

                    new_A[:, a, b] = (gamma.T * count_per_seq).sum(axis=1)

            # Normalize with smoothing
            new_A += self.smoothing
            row_sums = new_A.sum(axis=2, keepdims=True)
            new_A /= row_sums
            self.log_A = np.log(new_A)

        self.final_ll_ = log_lik_history[-1]
        self.gamma_ = gamma
        return self

    def get_params_exp(self):
        return np.exp(self.log_omega), np.exp(self.log_pi), np.exp(self.log_A)

# Module 3: Multi-Start Driver

def run_full_marks_solution():
    # 1. Load Data
    data = load_weather_data('meteo1.csv')
    if data is None: return

    # 2. Multi-Start Strategy
    n_restarts = 20
    best_model = None
    best_ll = -np.inf

    print(f"\nRunning Multi-Start EM ({n_restarts} restarts) to find global optimum...")
    print("-" * 60)
    print(f"{'Run ID':<10} | {'Log-Likelihood':<20} | {'Status'}")
    print("-" * 60)

    for i in range(n_restarts):

        model = MixtureMarkovEM(n_components=3, n_states=3, max_iter=100,
                                tol=1e-6, random_state=i, smoothing=1e-5)
        model.fit(data)

        is_best = False
        if model.final_ll_ > best_ll:
            best_ll = model.final_ll_
            best_model = model
            is_best = True

        mark = "*" if is_best else ""
        print(f"{i:<10} | {model.final_ll_:<20.4f} | {mark}")

    print("-" * 60)
    print(f"Best Log-Likelihood found: {best_ll:.4f}")

    # 3. Retrieve Best Parameters
    omega, pi, A = best_model.get_params_exp()
    gamma = best_model.gamma_

    # 4. Final Report (Exactly as required)
    print("\n" + "="*50)
    print(f"FINAL SUBMISSION RESULTS (Part b)")
    print("="*50)
    print(f"Log-Likelihood: {best_ll:.4f}")

    print(" Learned Parameters")

    print("\n[Mixture Weights (Omega)]")

    print(np.round(omega, 4))

    print("\n[Initial Probabilities (Pi)]")
    for k in range(3):
        print(f"  Station {k+1}: {np.round(pi[k], 4)}")

    print("\n[Transition Matrices (A)]")
    for k in range(3):
        print(f"\n  Station {k+1} (Row=From, Col=To):")
        print(np.round(A[k], 4))

    print("\n" + "="*65)
    print("Posterior Distribution (P(Z=k|X)) for First 10 Sequences")
    print("="*65)
    print(f"{'Seq ID':<8} | {'Station 1':<15} | {'Station 2':<15} | {'Station 3':<15}")
    print("-" * 65)

    for i in range(10):
        probs = gamma[i]
        probs /= probs.sum() # Normalize for display
        print(f"{i+1:<8} | {probs[0]:.6f}        | {probs[1]:.6f}        | {probs[2]:.6f}")

if __name__ == "__main__":
    run_full_marks_solution()